# B2.6 · Sandbox replication

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline, Before and After Deploy**  ·  *Security of AI*

Builds on **[B2.5 · Feasibility filtering, reachability and dead code](https://spbreed.github.io/cyber-commons/lessons/B2.5.html)**.

| | |
|---|---|
| Tools used | Docker, gVisor, Cilium, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Stand up an isolated replica, prove egress and credential isolation, and show what a destructive probe touches.

**Why a security engineer needs it.** Dynamic testing is run against staging, so a destructive probe becomes an incident. The control it builds is: stage 11: replicate the application in an isolated, disposable runtime with no path to production.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

You cannot exploit a finding to confirm it without somewhere safe to do it. The replica is that place, and the fidelity you give it decides which findings you are able to confirm at all.

> **At CyberTravels.** You cannot confirm the IDOR by exploiting it in production. The replica is where the booking API can be attacked safely, and its fidelity decides which findings are confirmable at all.

## 2 · The framework

```
   production            replica
   +-----------+         +------------------+
   | real data |   -->   | stubbed data     |
   | real deps |         | recorded deps    |
   | real users|         | nobody           |
   +-----------+         +------------------+
                                 |
                        exploit here, safely, on purpose

   fidelity decides which findings you can confirm at all
```

Phase 4 turns hypotheses into facts by running the application. That is only
safe if the thing you run it against cannot hurt anyone.

**Stage 11 — Sandbox replication.** Deploy the application in an isolated,
disposable runtime: its own container, its own synthetic data, no route to
production, no real credentials.

The reason this is a *stage* rather than a footnote is that the obvious shortcut
— point the dynamic tests at staging — converts every destructive probe into an
incident. Staging usually shares an identity provider, a message bus, sometimes
a database replica, and always someone's on-call rota.

Four isolation properties, and you need all four:

- **network** — no egress except to the replica itself,
- **credentials** — synthetic secrets, so a leak is worthless,
- **data** — synthetic records, so an exfiltration test exfiltrates nothing,
- **lifetime** — destroyed after the run, so state cannot leak between tests.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · The stage, as a skill

Before anything is executed against CyberTravels' environment, four checks decide whether it is a replica or staging with a different DNS name. The skill runs them — egress, credentials, data, and the destructive probes you would only run somewhere built to be destroyed.

### The skill — [`skills/appsec/exploit-replica-isolation-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/exploit-replica-isolation-check/SKILL.md)

```yaml
name: exploit-replica-isolation-check
description: >-
  Check that the environment a finding is proved in is a replica rather than
  staging — on egress, credentials, data realism and destructive probes — before
  anything is executed against it. Use before dynamic testing, exploit
  validation, or letting an agent run a proof of concept.
allowed-tools: Read, Grep, Glob, Bash
```

# Staging is production with a different DNS name

The reason exploitation is safe is the environment, and "staging" is not that
environment: it holds real credentials, real-shaped customer records, and a
route to things that matter. A replica is built to be destroyed, and the four
checks that distinguish them take minutes.

## When to use this

Before running any probe that could change state, before pointing an agent at a
target, and every time somebody offers staging because the replica is not ready.

## Procedure

**1 — Check egress.** The environment should reach its own internal hosts and
nothing else. Test the three that matter specifically: a code host, the cloud
metadata address, and any private range. A replica that can reach GitHub can
exfiltrate.

**2 — Check credentials.** Every secret in the environment should be synthetic.
Grep the environment, the mounted files and the database. One real key makes the
whole environment production for the purposes of this decision.

**3 — Check the data.** Real-shaped is not the same as real. Sample records and
confirm they are generated, not copied — a customer row with a real email
address is a breach waiting for a log line.

**4 — Run the destructive probes.** Drop a table, delete a file, exhaust a
quota. In a replica these are unremarkable. If you would not do them here, this
is not the environment to prove an exploit in, and that is the finding.

**5 — Record the verdict per check, not overall.** "Isolated: no" sends people
looking; "credentials: fail, egress: pass" tells them what to fix.

## Example

**Input** — the fixture committed at the top of [`scripts/exploit_replica_isolation_check.py`](scripts/exploit_replica_isolation_check.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
ALLOW http://replica.local/reports                   replica-internal
ALLOW http://db.replica.local:5432/                  replica-internal
DENY  https://api.github.com/                        not on the replica allowlist
DENY  http://169.254.169.254/latest/meta-data/       private address outside the replica — blocked
DENY  http://10.0.3.14:9200/_search                  private address outside the replica — blocked
probe                       on staging    on replica
--------------------------------------------------------
drop a table                   REACHES     contained
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "environment": "str",
  "checks": {"egress": {"pass": false, "reachable": ["str"]},
             "credentials": {"pass": false, "real_found": ["str"]},
             "data": {"pass": false, "sample_real": ["str"]},
             "destructive": {"pass": false, "refused": ["str"]}},
  "verdict": "replica|not a replica",
  "safe_to_exploit": false
}
```

## Failure modes

- **Accepting the name.** "staging" and "replica" are labels; the four checks
  are the definition.
- **Testing general egress only.** The metadata address is the one that turns a
  probe into a credential theft.
- **Skipping the destructive probes** because they are destructive. That
  reluctance is the answer.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/exploit-replica-isolation-check/scripts/exploit_replica_isolation_check.py
SCRIPT = "skills/appsec/exploit-replica-isolation-check/scripts/exploit_replica_isolation_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The replica permits only its own internal hosts and blocks GitHub, the metadata service and private addresses. Staging holds real credentials and a real-shaped customer record while the replica holds synthetic ones. The four isolation checks pass for the replica and fail for staging on credentials, data and lifetime, and destroying the replica clears its state.

## Your turn

Check whether your dynamic testing currently runs against staging. If it does, list what staging shares with production — identity provider, message bus, data replica. Each shared component is a path from a test probe to a real incident.

---

**Next → [B2.7 · Supply chain — SBOM, dependency vulnerabilities, and decompiling the libraries](https://spbreed.github.io/cyber-commons/lessons/B2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*